# Notebook 1: cleaning

Part 1 (Volume, Spark SQL) of our project: how do rain, snow and temperature change trips, trip time and fares per zone?

This notebook cleans the weather and the trips. Run `scripts/download_data.py` and `src/ingest.py` first (see the README). Then run the cells one by one from top to bottom (or use *Run > Run All Cells*). It takes about 10 minutes.

- **Part A:** clean the weather
- **Part B:** clean the trips

It writes `data/parquet/weather_clean`, `data/parquet/trips_clean` and the row counts in `outputs/cleaning/`. The next notebook, `02_analysis.ipynb`, uses them.

## Setup

In [1]:
import os

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# the notebook starts in notebooks/, go to the project folder so the paths below work
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

OUT = "data/parquet"
os.makedirs("outputs/cleaning", exist_ok=True)

spark = (
    SparkSession.builder.master("local[*]").appName("clean and analyze")
    .config("spark.ui.showConsoleProgress", "false")  # no progress bars in the output
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("spark", spark.version, "| project folder:", os.getcwd())

spark 3.5.0 | project folder: /home/jovyan/work


Two small helpers for the cleaning steps. `log_step` counts the rows after a step and prints how many were removed. `save_counts` saves all steps as a csv file for the report.

In [2]:
def log_step(df, name, counts):
    print(f"{name} ... ", end="", flush=True)  # shown while spark is counting
    left = df.count()
    removed = counts[-1][1] - left
    counts.append((name, left))
    print(f"{left} left, {removed} removed")
    return df


def save_counts(counts, path):
    rows = []
    for i, (step, left) in enumerate(counts):
        removed = 0 if i == 0 else counts[i - 1][1] - left
        rows.append({"step": step, "rows_after": left, "rows_removed": removed})
    table = pd.DataFrame(rows)
    table.to_csv(path, index=False)
    return table


print("helpers ready")

helpers ready


## Part A: clean the weather

### Step 1: load the weather

This is the table from `ingest.py`: one row per day for our 915 days. Rain (`prcp`) and temperature are still in tenths, snow is in mm.

In [3]:
weather = spark.read.parquet(f"{OUT}/weather")
weather_counts = [("weather start", weather.count())]
print("weather start:", weather_counts[0][1], "rows")
weather.limit(3).toPandas()

weather start: 915 rows


,date,prcp,snow,tmax,tmin
0,2023-07-01,0,0,278,206
1,2023-07-02,147,0,311,211
2,2023-07-03,114,0,317,222


### Step 2: missing values

We drop the rows that have a missing value.

In [4]:
weather = log_step(weather.dropna(), "drop rows with missing values", weather_counts)

drop rows with missing values ... 915 left, 0 removed


### Step 3: duplicate days

We want one row per day.

In [5]:
weather = log_step(weather.dropDuplicates(["date"]), "drop duplicate days", weather_counts)

drop duplicate days ... 915 left, 0 removed


### Step 4: units

NOAA gives rain and temperature in tenths. We convert to mm and celsius and give the columns clear names.

In [6]:
weather = weather.select(
    "date",
    (F.col("prcp") / 10).alias("prcp_mm"),
    F.col("snow").alias("snow_mm"),
    (F.col("tmax") / 10).alias("tmax_c"),
    (F.col("tmin") / 10).alias("tmin_c"),
)
weather.limit(3).toPandas()

,date,prcp_mm,snow_mm,tmax_c,tmin_c
0,2023-07-15,1.0,0,31.1,21.7
1,2024-09-18,0.0,0,23.9,19.4
2,2025-02-16,19.6,0,8.3,0.6


### Step 5: impossible values

Negative rain or snow, a max temperature above 50 C, a min temperature below -40 C, or a min temperature above the max.

In [7]:
bad = "prcp_mm < 0 OR snow_mm < 0 OR tmax_c > 50 OR tmin_c < -40 OR tmin_c > tmax_c"
weather = log_step(weather.where(f"NOT ({bad})"), "drop impossible values", weather_counts)

drop impossible values ... 915 left, 0 removed


### Step 6: save the clean weather

Nothing had to be removed, so 915 days stay 915 days.

In [8]:
print("saving the clean weather ...")
weather.write.mode("overwrite").parquet(f"{OUT}/weather_clean")
_ = save_counts(weather_counts, "outputs/cleaning/weather.csv")
print("saved data/parquet/weather_clean and outputs/cleaning/weather.csv")

saving the clean weather ...
saved data/parquet/weather_clean and outputs/cleaning/weather.csv


## Part B: clean the trips

We found these problems by counting them on all 108.7 million trips: wrong dates, zero or negative trip times, negative or zero fares, zero distances, unknown zones and a few duplicates. Every step below removes one kind of problem, and the output shows how many rows it removed.

### Step 7: load the trips

This is the table from `ingest.py`. We add the trip time in minutes.

In [9]:
trips = spark.read.parquet(f"{OUT}/trips")
trips = trips.withColumn(
    "duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60,
)
trip_counts = [("trips start", trips.count())]
print("trips start:", trip_counts[0][1], "rows")

trips start: 108708928 rows


### Step 8: pick up date

Only trips from July 2023 to December 2025. The raw files also contain a few trips with dates from 2002 to 2009 and from 2026.

In [10]:
trips = log_step(
    trips.where("tpep_pickup_datetime >= '2023-07-01' AND tpep_pickup_datetime < '2026-01-01'"),
    "pickup date inside jul 2023 - dec 2025", trip_counts)

pickup date inside jul 2023 - dec 2025 ... 108708813 left, 115 removed


### Step 9: trip time

Between 1 and 180 minutes. This removes negative times (drop off before pick up), zero times, trips under one minute and trips of more than 3 hours.

In [11]:
trips = log_step(trips.where("duration_min >= 1 AND duration_min <= 180"),
                 "duration between 1 and 180 min", trip_counts)

duration between 1 and 180 min ... 106832877 left, 1875936 removed


### Step 10: fare

More than 0 and at most 500 dollars. Negative fares are credits and adjustments (flex fare, dispute, cash, no charge), not normal paid trips.

In [12]:
trips = log_step(trips.where("fare_amount > 0 AND fare_amount <= 500"),
                 "fare above 0 and max 500", trip_counts)

fare above 0 and max 500 ... 103210679 left, 3622198 removed


### Step 11: distance

More than 0 and at most 100 miles.

In [13]:
trips = log_step(trips.where("trip_distance > 0 AND trip_distance <= 100"),
                 "distance above 0 and max 100 miles", trip_counts)

distance above 0 and max 100 miles ... 101616892 left, 1593787 removed


### Step 12: zones

The analysis is per pick up zone, so we remove pick ups from zone 264 (Unknown) and 265 (Outside of NYC). For the drop off we only remove 264, because trips that end outside NYC are real.

In [14]:
trips = log_step(trips.where("PULocationID NOT IN (264, 265)"),
                 "pickup zone known (not 264, 265)", trip_counts)

pickup zone known (not 264, 265) ... 101287309 left, 329583 removed


In [15]:
trips = log_step(trips.where("DOLocationID != 264"),
                 "dropoff zone known (not 264)", trip_counts)

dropoff zone known (not 264) ... 101088750 left, 198559 removed


### Step 13: duplicates and columns

We remove exact duplicate rows (a hash of the whole original row, we keep one of each) and keep only the 11 columns we need. Trips without a passenger count (13 million) are kept, we never use that column. Then we save the clean trips.

In [16]:
print("removing duplicates and saving the clean trips (takes about a minute) ...")
all_columns = [c for c in trips.columns if c != "duration_min"]
trips = trips.withColumn("row_hash", F.xxhash64(*all_columns)).dropDuplicates(["row_hash"])

trips = trips.select(
    F.col("tpep_pickup_datetime").alias("pickup_ts"),
    F.col("tpep_dropoff_datetime").alias("dropoff_ts"),
    F.to_date("tpep_pickup_datetime").alias("pickup_date"),
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "duration_min",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "passenger_count",
)
trips.write.mode("overwrite").parquet(f"{OUT}/trips_clean")
print("saved data/parquet/trips_clean")

removing duplicates and saving the clean trips (takes about a minute) ...
saved data/parquet/trips_clean


In [17]:
trips = log_step(spark.read.parquet(f"{OUT}/trips_clean"), "drop duplicate rows", trip_counts)

drop duplicate rows ... 101088746 left, 4 removed


### Step 14: all cleaning steps

Rows left and removed after every step, for the weather and for the trips (also saved in `outputs/cleaning/`).

In [18]:
pd.concat([
    save_counts(weather_counts, "outputs/cleaning/weather.csv"),
    save_counts(trip_counts, "outputs/cleaning/trips.csv"),
], ignore_index=True)

,step,rows_after,rows_removed
0,weather start,915,0
1,drop rows with missing values,915,0
2,drop duplicate days,915,0
3,drop impossible values,915,0
4,trips start,108708928,0
5,pickup date inside jul 2023 - dec 2025,108708813,115
6,duration between 1 and 180 min,106832877,1875936
7,fare above 0 and max 500,103210679,3622198
8,distance above 0 and max 100 miles,101616892,1593787
9,"pickup zone known (not 264, 265)",101287309,329583
